# Parts B and D: ResNet-18 transfer learning

Keep this file beside `part_a.ipynb` and run top to bottom.
Part A supplies the data and preprocessing; this notebook trains and compares a frozen backbone with a fine-tuned last block.
Checkpoints are selected by validation macro-F1. Parts C/E use the exported models and histories.

### Setup: load Part A

Takes the existing Part A file or its variables in this kernel.
Provides loaders, class names and preprocessing settings.

In [ ]:
from pathlib import Path

PART_A_PATH = Path("part_a.ipynb")
required_a = ("train_loader", "val_loader", "NUM_CLASSES", "CLASS_NAMES", "IMAGE_SIZE",
              "SEED", "IMAGENET_MEAN", "IMAGENET_STD")
if not all(name in globals() for name in required_a):
    if not PART_A_PATH.is_file():
        raise FileNotFoundError("Place part_a.ipynb here or update PART_A_PATH")
    get_ipython().run_line_magic("run", f'"{PART_A_PATH.resolve().as_posix()}"')
missing = [name for name in required_a if name not in globals()]
if missing:
    raise RuntimeError("Missing Part A outputs: " + ", ".join(missing))
class_names = list(CLASS_NAMES)
assert train_loader.num_workers == 0, "Use num_workers=0 for reproducible branch comparisons"

### Settings

Takes class count, image size and seed from Part A.
Provides learning rates, epoch counts and output path.

In [ ]:
CFG = {
    "image_size": IMAGE_SIZE, "num_classes": NUM_CLASSES,
    "seed": SEED, "head_epochs": 5, "stage2_epochs": 5,
    "head_lr": 1e-3, "finetune_head_lr": 1e-4, "backbone_lr": 1e-5,
    "weight_decay": 1e-4,
    "output_dir": Path("artifacts/part_b_d"),
}
assert CFG["num_classes"] == len(class_names)
assert CFG["head_epochs"] > 0 and CFG["stage2_epochs"] > 0

### Reproducibility

Takes CFG and the training loader.
Provides device and seed-reset helpers.

In [ ]:
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision import models
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def reset_train_seed(loader, seed):
    seed_everything(seed)
    for generator in (loader.generator, getattr(loader.sampler, "generator", None)):
        if generator is not None:
            generator.manual_seed(seed)

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


### B1. Replace the classifier

Takes the class count and ImageNet weights.
Provides ResNet-18 with a new final layer.

The output is raw logits [B, K]; CrossEntropyLoss does not need a separate Softmax.

In [ ]:
def build_model(num_classes, weights):
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

seed_everything(CFG["seed"])
model = build_model(CFG["num_classes"], models.ResNet18_Weights.DEFAULT).to(device)
print(model.fc)

Linear(in_features=512, out_features=10, bias=True)


### B2. Freeze and count parameters

Takes model and the stage name.
Provides stage controls and trainable parameter counts.

Stage 1 trains fc; stage 2 trains layer4 + fc.

In [ ]:
def set_stage(model, stage):
    if stage not in ("head", "finetune"):
        raise ValueError("stage must be head or finetune")
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.fc.parameters():
        parameter.requires_grad = True
    if stage == "finetune":
        for parameter in model.layer4.parameters():
            parameter.requires_grad = True

def count_parameters(model):
    return {"total": sum(p.numel() for p in model.parameters()),
            "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}

parameter_rows = []
for stage in ("head", "finetune"):
    set_stage(model, stage)
    parameter_rows.append({"stage": stage, **count_parameters(model)})
parameter_table = pd.DataFrame(parameter_rows)
display(parameter_table)
set_stage(model, "head")

,stage,total,trainable
0,head,11181642,5130
1,finetune,11181642,8398858


### B3. Training and validation

Takes model, loader, loss, optimizer and device.
Provides train_epoch and evaluate.

loss.backward() computes gradients; optimizer.step() updates weights. evaluate uses inference_mode and never updates weights.

In [ ]:
def classification_metrics(y_true, y_pred, num_classes):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=range(num_classes),
                                    average="macro", zero_division=0)),
    }

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()

    # Keep running statistics fixed in frozen BatchNorm layers.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            if not any(p.requires_grad for p in module.parameters()):
                module.eval()
    total_loss, total = 0.0, 0
    y_true, y_pred = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()  # Compute gradients.
        optimizer.step()  # Update weights.
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total += batch_size
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(logits.detach().argmax(dim=1).cpu().tolist())
    if total == 0:
        raise ValueError("Empty training loader")
    return {"loss": total_loss / total,
            **classification_metrics(y_true, y_pred, logits.shape[1])}

@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss, total = 0.0, 0
    y_true, y_pred = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item()
        total += labels.size(0)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    if total == 0:
        raise ValueError("Empty evaluation loader")
    return {"loss": total_loss / total,
            **classification_metrics(y_true, y_pred, logits.shape[1])}

### B4. Select checkpoints

Takes training/validation functions and stage settings.
Provides history and the best validation checkpoint.

State tensors are cloned so later training cannot overwrite the best checkpoint.

In [ ]:
def snapshot(model):

    return {name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()}

def fit_stage(model, train_loader, val_loader, criterion, optimizer, device,
              epochs, stage, epoch_offset=0, initial_best=None):
    history = []
    best = copy.deepcopy(initial_best)
    for local_epoch in range(1, epochs + 1):
        train = train_epoch(model, train_loader, criterion, optimizer, device)
        val = evaluate(model, val_loader, device)
        row = {"epoch": epoch_offset + local_epoch, "stage": stage,
               **{f"train_{k}": v for k, v in train.items()},
               **{f"val_{k}": val[k] for k in ("loss", "accuracy", "macro_f1")}}
        history.append(row)
        if best is None or val["macro_f1"] > best["score"]:
            best = {"score": val["macro_f1"], "epoch": row["epoch"],
                    "stage": stage, "state": snapshot(model)}
        print(f"{stage:12s} {row['epoch']:02d} | "
              f"loss {train['loss']:.3f}/{val['loss']:.3f} | "
              f"acc {train['accuracy']:.3f}/{val['accuracy']:.3f} | "
              f"val F1 {val['macro_f1']:.3f}")
    model.load_state_dict(best["state"])
    return history, best

### B5. Train the head

Takes the frozen backbone and Part A loaders.
Provides head_best, shared by both experiment branches.

In [ ]:
reset_train_seed(train_loader, CFG["seed"])
set_stage(model, "head")
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG["head_lr"], weight_decay=CFG["weight_decay"])
head_history, head_best = fit_stage(
    model, train_loader, val_loader, criterion, optimizer, device,
    epochs=CFG["head_epochs"], stage="head")
print("Best head epoch:", head_best["epoch"], "val F1:", head_best["score"])

head         01 | loss 2.523/2.352 | acc 0.120/0.100 | val F1 0.095
head         02 | loss 2.127/2.172 | acc 0.220/0.190 | val F1 0.152
head         03 | loss 1.932/1.975 | acc 0.350/0.320 | val F1 0.329
head         04 | loss 1.728/1.776 | acc 0.550/0.500 | val F1 0.469
head         05 | loss 1.490/1.616 | acc 0.700/0.590 | val F1 0.533
Best head epoch: 5 val F1: 0.5332942678236796


### B6. Fine-tune layer4

Takes head_best and the same loaders.
Provides finetuned_model and its best checkpoint.

A new optimizer includes layer4; its learning rate is lower than the head's.

In [ ]:
model.load_state_dict(head_best["state"])
set_stage(model, "finetune")

optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": CFG["backbone_lr"]},
    {"params": model.fc.parameters(), "lr": CFG["finetune_head_lr"]},
], weight_decay=CFG["weight_decay"])
reset_train_seed(train_loader, CFG["seed"] + 1)
ft_history, ft_best = fit_stage(
    model, train_loader, val_loader, criterion, optimizer, device,
    epochs=CFG["stage2_epochs"], stage="finetune",
    epoch_offset=CFG["head_epochs"], initial_best=head_best)
finetuned_model = model
print("Stage 2 trainable parameters:", count_parameters(model)["trainable"])
print("Best checkpoint:", ft_best["stage"], ft_best["epoch"], ft_best["score"])

finetune     06 | loss 1.618/1.553 | acc 0.580/0.650 | val F1 0.594
finetune     07 | loss 1.439/1.493 | acc 0.670/0.660 | val F1 0.611
finetune     08 | loss 1.300/1.435 | acc 0.780/0.680 | val F1 0.630
finetune     09 | loss 1.229/1.363 | acc 0.780/0.700 | val F1 0.655
finetune     10 | loss 1.106/1.302 | acc 0.820/0.730 | val F1 0.682
Stage 2 trainable parameters: 8398858
Best checkpoint: finetune 10 0.682464482044692


### D1. Frozen control

Takes the same head_best and additional epoch budget.
Provides frozen_model and its best checkpoint.

Both branches use the same starting weights, head learning rate, batch order, augmentations and epochs; only layer4 training differs.

In [ ]:
frozen_model = build_model(CFG["num_classes"], weights=None).to(device)
frozen_model.load_state_dict(head_best["state"])
set_stage(frozen_model, "head")

frozen_optimizer = torch.optim.AdamW(
    [p for p in frozen_model.parameters() if p.requires_grad],
    lr=CFG["finetune_head_lr"], weight_decay=CFG["weight_decay"])
reset_train_seed(train_loader, CFG["seed"] + 1)
frozen_history, frozen_best = fit_stage(
    frozen_model, train_loader, val_loader, criterion, frozen_optimizer, device,
    epochs=CFG["stage2_epochs"], stage="frozen",
    epoch_offset=CFG["head_epochs"], initial_best=head_best)

frozen       06 | loss 1.351/1.585 | acc 0.760/0.630 | val F1 0.573
frozen       07 | loss 1.325/1.560 | acc 0.740/0.660 | val F1 0.604
frozen       08 | loss 1.303/1.538 | acc 0.840/0.700 | val F1 0.643
frozen       09 | loss 1.280/1.518 | acc 0.810/0.730 | val F1 0.675
frozen       10 | loss 1.266/1.500 | acc 0.860/0.730 | val F1 0.688


### D2. Compare on validation

Takes both best checkpoints and the validation loader.
Provides comparison, histories, candidates and selected_model.

Fine-tuning can adapt features but may overfit a small training set; augmentation provides regularization. Explain the observed table, then let Part C evaluate the fixed selection on test.

In [ ]:
candidates = {
    "frozen": {"model": frozen_model, "best": frozen_best},
    "finetune": {"model": finetuned_model, "best": ft_best},
}
histories = {
    "frozen": pd.DataFrame(head_history + frozen_history),
    "finetune": pd.DataFrame(head_history + ft_history),
}
rows = []
for name, item in candidates.items():
    metrics = evaluate(item["model"], val_loader, device)
    rows.append({"setting": name, "trainable_stage2": count_parameters(item["model"])["trainable"],
                 "best_epoch": item["best"]["epoch"], "best_stage": item["best"]["stage"],
                 "val_loss": metrics["loss"], "val_accuracy": metrics["accuracy"],
                 "val_macro_f1": metrics["macro_f1"]})
comparison = pd.DataFrame(rows)
display(comparison.round(4))

selected_name = max(candidates, key=lambda name: candidates[name]["best"]["score"])
selected_model = candidates[selected_name]["model"]
print("Selected by validation:", selected_name)

,setting,trainable_stage2,best_epoch,best_stage,val_loss,val_accuracy,val_macro_f1
0,frozen,5130,10,frozen,1.5001,0.73,0.6884
1,finetune,8398858,10,finetune,1.3020,0.73,0.6825


Selected by validation: frozen


### D3. Export for Parts C/E

Takes models, histories and Part A preprocessing.
Provides state_dict files, CSV histories and metadata.

In [ ]:
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
comparison.to_csv(CFG["output_dir"] / "comparison_validation.csv", index=False)
for name, item in candidates.items():
    torch.save(snapshot(item["model"]), CFG["output_dir"] / f"{name}_state.pt")
    histories[name].to_csv(CFG["output_dir"] / f"history_{name}.csv", index=False)
metadata = {
    "architecture": "resnet18", "num_classes": CFG["num_classes"],
    "in_channels": 3, "image_size": CFG["image_size"],
    "class_names": class_names, "mean": list(IMAGENET_MEAN), "std": list(IMAGENET_STD),
    "data_pipeline": "part_a.ipynb", "batch_size": train_loader.batch_size,
    "selected_setting": selected_name, "selection_metric": "val_macro_f1",
    "checkpoints": {name: {"file": f"{name}_state.pt", "epoch": item["best"]["epoch"],
                           "stage": item["best"]["stage"], "val_macro_f1": item["best"]["score"]}
                    for name, item in candidates.items()},
    "seed": CFG["seed"],
}
(CFG["output_dir"] / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved to:", CFG["output_dir"].resolve())

Saved to: C:\Users\bbdng\Desktop\base\Projects\Final_CV\artifacts\part_b_d
